# Analysis of swim plate density profiles for simulation comparision

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import simpson
import glob
import os
from Exper_profileprocess import BiomassDynamics, ProfileMultiTime, BiomassEndpoint
curr_dir=os.getcwd()

# Timelapse Data

In [3]:
os.chdir("path/to/Swimplate_Data/Converted-density-profiles/")

biom_str=['WT_025ag','WT_028ag','del metH']
biom_exper=[glob.glob('path/to/Swimplate_Data/Converted-density-profiles/'+f'/Density-profile*{x}*Tlapse.xlsx')[0] for x in biom_str]
biom_exper
swmplt_data=BiomassDynamics(biom_exper, time_offset=6)

radii_dt=swmplt_data.radii_dt() # spreading distance versus time
bulkbiom_dt=swmplt_data.bulkavg_dt() # bulk biomass growth over time

t_vis=[10,13,16,18,20]
profile_exper=ProfileMultiTime(biom_exper, time_offset=6, tpoints_j=t_vis)
prof_scaling=profile_exper.bulkavg_dt()
li_scaled, avg_norm, std_norm=profile_exper.densityprofile_compile() # density profiles scaled to maximum bulk biomass density

In [10]:
os.chdir('path/to/Swimplate_Data/Macrocolony-analysis/Timelapse')
for j in range(len(li_scaled)):
    with pd.ExcelWriter(f'Density-profile_scaled_Replicate-{biom_str[j]}.xlsx') as writer:
        for k in range(len(li_scaled[j])):
            li_scaled[j][k].to_excel(writer,sheet_name=f'{t_vis[k]}_hour')
            
with pd.ExcelWriter("Bulk-biomass-dynamics_scaled.xlsx") as writer:
    for j in range(len(biom_str)):
        bulkbiom_dt[j].to_excel(writer,sheet_name='Bulk_Biomass_'+biom_str[j])
        
with pd.ExcelWriter("Spreading-distance_Replicates.xlsx") as writer:
    for j in range(len(biom_str)):
        radii_dt[j].to_excel(writer,sheet_name='Colony_Radius_'+biom_str[j])
        
with pd.ExcelWriter("Density-profile_scaled_Group-Avg.xlsx") as writer:
    for j in range(len(biom_str)):
        avg_norm[j].to_excel(writer,sheet_name='Profile_'+biom_str[j])

with pd.ExcelWriter("Density-profile_scaled_Group-StDev.xlsx") as writer:
    for j in range(len(biom_str)):
        std_norm[j].to_excel(writer,sheet_name='Profile_'+biom_str[j])
        
g=pd.DataFrame(np.array(swmplt_data.cb_bulkavg),columns=[f'Rep_{x+1}' for x in range(len(swmplt_data.sheets_exper[0]))])
g['Group']=biom_str
g=g.set_index('Group')
g.to_csv("Density-profile_gDW-scaling-factor.csv")

# Endpoint

In [4]:
os.chdir("path/to/Swimplate_Data/Converted-density-profiles")
biom_exper=[glob.glob('path/to/Swimplate_Data/Converted-density-profiles/'+f'/Density-profile*Endpt.xlsx')[0]]
biom_exper

['D:/Simul_GEMCM/Experiment_meas\\Density-profile-avg_Endpt.xlsx']

In [5]:
swmplt_data2=BiomassEndpoint(biom_exper)

profile_endpt, profile_avg, profile_std=swmplt_data2.densityprofile_() # compile average and standard deviation

In [9]:
os.chdir("path/to/Swimplate_Data/Macrocolony-analysis/Single-tpoint")
profile_avg.to_csv("Density-profile_Replicate_gDW_Endpt.csv")
profile_std.to_csv("Density-profile_StDev_gDW_Endpt.csv")